# Single Cell data
CC 2026-08-11

## 1. Setup

In [ ]:
import retinanalysis as ra
import matplotlib.pyplot as plt
import numpy as np

# Read-only single-cell database queries and notebook browsers.
from retinanalysis.SCutils import explore as sc
# Map new h5 files when needed.
report = ra.SCutils.update_single_cell_json()

## 2. Populate and refresh the database

`populate_database()` ingests new experiments, refreshes experiments whose JSON changed, and returns the database freshness check in the same report. Canonical experiment names such as `YYYY-MM-DD_X.h5` are included; auxiliary and legacy files are ignored. By default only metadata and tags JSON files trigger a refresh. Pass `watch_data_file=True` to include H5 modification times.

In [ ]:
# One call handles ingest, refresh, and the post-ingest stale-file check.
summary = ra.populate_database()
df_db = summary['experiments']
df_stale = summary['stale']

print(f"newly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")
print(f"database    : {len(df_db)} experiments; {len(df_stale)} still out of date")

if len(df_stale):
    display(df_stale[['exp_name', 'date_added', 'source_mtime', 'source_file']])

# ra.purge_experiments('2026-06-04_G')
# ra.purge_experiments(['2026-05-06_E', '2026-05-08_E'])

# Drop only rows that remain stale after populate.
# ra.purge_experiments(df_stale['exp_name'].tolist())

# Wipe the whole database (requires the literal confirmation token).
# ra.purge_database(confirm='YES_DELETE_ALL')

print(f'{len(df_db)} experiments currently in the database.')

## 3. List single-cell experiments

Experiments are shown in separate `chris_data` and `fred_data` tables. Each protocol gets its own row so a date is easier to scan; repeated experiment, project, and short cell-type values are visually grouped. Owner and species are kept only for the cascading browser and are omitted from the table.

In [8]:
# Section 3 has exactly these visible columns. Owner and species remain
# internal to the cascading browser and are not included here.
df_sc_exps = sc.list_experiments(show=False)

# Accept both the current singular column and an older, comma-joined
# `protocols` column from a module already cached in this kernel.
if 'protocols' in df_sc_exps.columns:
    df_sc_exps['protocol'] = df_sc_exps.pop('protocols').fillna('?').str.split(r',\s*', regex=True)
    df_sc_exps = df_sc_exps.explode('protocol', ignore_index=True)

section3_columns = ['exp_name', 'cell_types', 'protocol']
df_sc_exps = (df_sc_exps.loc[:, section3_columns]
              .drop_duplicates()
              .sort_values(['exp_name', 'protocol'], ignore_index=True))
sc.tree_table(df_sc_exps, levels=['exp_name', 'project', 'cell_types'], height=500)

exp_name,cell_types,protocol
2017-11-21_B,"OFF-transient, ON-alpha",ExpandingSpots
,,EyeMovementTrajectory
2017-12-12_B,"AII, rod bipolar",LedPulse
,,LedPulseFamily
2018-09-06_B,ON-alpha,ChirpStimulus
,,ChirpStimulusLED
,,LedPulse
2018-10-05_B,?,SingleSpot
2019-01-08_B,unknown,LedPulse
,,PulseFamily


'\n<style>\n.ra-tbl { overflow: auto; }\n.ra-tbl table { border-collapse: collapse; font-size: 12.5px;\n                font-variant-numeric: tabular-nums; }\n.ra-tbl th { position: sticky; top: 0; z-index: 1; text-align: left;\n             font-weight: 600; padding: 4px 12px 4px 0;\n             border-bottom: 1px solid rgba(128,128,128,0.6);\n             background: var(--jp-layout-color0, #fff); }\n.ra-tbl td { padding: 2px 12px 2px 0; vertical-align: top;\n             white-space: nowrap; }\n.ra-tbl tr.grp > td { border-top: 1px solid rgba(128,128,128,0.28); }\n.ra-tbl td.num { text-align: right; }\n.ra-tbl td.lead { font-weight: 600; }\n.ra-tbl summary { cursor: pointer; margin: 2px 0; }\n</style>\n<div class="ra-tbl" style="max-height:500px"><table><thead><tr><th>exp_name</th><th>cell_types</th><th>protocol</th></tr></thead><tbody><tr><td class="lead">2017-11-21_B</td><td>OFF-transient, ON-alpha</td><td>ExpandingSpots</td></tr><tr><td class="lead"></td><td></td><td>EyeMovement

## 4. Find experiments by protocol

Search protocol names case-insensitively. The returned DataFrame remains one row per epoch block.

In [9]:
df_blocks = sc.find_blocks('spotWithAnnularContrastReversingGrating')

107 blocks | 9 experiments | 1 protocol(s) matching 'spotWithAnnularContrastReversingGrating'


exp_name,blocks,protocols,block_ids
2026-04-23_E,17,spotWithAnnularContrastReversingGrating,"34281, 34284-34286, 34291, 34294-34295, 34305-34306, 34309, 34315, 34317, 34324-34325, 34330, 34333-34334"
2026-04-24_E,2,spotWithAnnularContrastReversingGrating,34360-34361
2026-04-28_E,7,spotWithAnnularContrastReversingGrating,"33036-33037, 33041-33042, 33044, 33090-33091"
2026-05-06_E,1,spotWithAnnularContrastReversingGrating,36399
2026-05-27_G,15,spotWithAnnularContrastReversingGrating,"37167-37172, 37178-37179, 37191-37197"
2026-06-02_G,20,spotWithAnnularContrastReversingGrating,"37214-37221, 37227-37234, 37254-37257"
2026-06-04_G,8,spotWithAnnularContrastReversingGrating,"35019-35022, 35031-35034"
2026-07-15_G,11,spotWithAnnularContrastReversingGrating,"37279-37282, 37285-37291"
2026-08-04_G,26,spotWithAnnularContrastReversingGrating,"37361-37380, 37382-37387"


exp_name,protocol,block_id
2026-04-23_E,spotWithAnnularContrastReversingGrating,34281
2026-04-23_E,spotWithAnnularContrastReversingGrating,34284
2026-04-23_E,spotWithAnnularContrastReversingGrating,34285
2026-04-23_E,spotWithAnnularContrastReversingGrating,34286
2026-04-23_E,spotWithAnnularContrastReversingGrating,34291
2026-04-23_E,spotWithAnnularContrastReversingGrating,34294
2026-04-23_E,spotWithAnnularContrastReversingGrating,34295
2026-04-23_E,spotWithAnnularContrastReversingGrating,34305
2026-04-23_E,spotWithAnnularContrastReversingGrating,34306
2026-04-23_E,spotWithAnnularContrastReversingGrating,34309


## 5. Browse and summarize experiments

Use the cascading menus to select data owner, species, and experiment. The overview is organized as cell → epoch group (group label) → protocol, with block and epoch counts. Then select an epoch block and click **Load original traces** to read and display every unprocessed Amp1 epoch trace from the H5 file.

In [10]:
experiment_browser = sc.summarize_experiments(df_sc_exps)